# MedAgentBench GRPO Training with Unsloth + TRL

Fine-tune **Qwen3-1.7B** on the MedAgentBench clinical decision-making benchmark using:
- **Unsloth** for 2-4x faster LoRA fine-tuning
- **TRL GRPOTrainer** with environment `tool-use` rollouts
- **Mock FHIR** backend — no live server, runs fully offline
- **Named tool calls** matching benchmark evaluation format

**Runtime:** T4 minimum, A100/H100 recommended

## 1. Install Dependencies & Clone Repo

In [2]:
# Install training stack
!pip install -q unsloth trl datasets huggingface_hub

# Clone repo (contains medagentbench_env + benchmark data)
!git clone https://github.com/ananya173147/clinKriya.git ./repo

# Verify GPU
!nvidia-smi | head -20

fatal: destination path './repo' already exists and is not an empty directory.
Sun Mar  8 17:16:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:DB:00.0 Off |                    0 |
| N/A   26C    P0             69W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                  

In [1]:
!git -C ./repo pull 

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.16 KiB | 32.00 KiB/s, done.
From https://github.com/ananya173147/clinKriya
   4ab5d0b..a4a2c4d  main       -> origin/main
Updating 4ab5d0b..a4a2c4d
Fast-forward
 medagentbench_env/train.py | 30 ++++++++++++++++++++++++++++--
 1 file changed, 28 insertions(+), 2 deletions(-)


## 2. Load Model with Unsloth (4-bit LoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType
import torch

MODEL_NAME = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Unsloth patches Qwen3 attention even for AutoModelForCausalLM.
# Its GRPO autocaster runs in fp16, so base weights + LoRA must also be fp16.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Load Environment & Data

In [3]:
import importlib.util, sys
from pathlib import Path

REPO = Path("./repo")

# Load train.py directly — avoids installing openenv-core
spec = importlib.util.spec_from_file_location(
    "train", REPO / "medagentbench_env" / "train.py"
)
train_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_mod)

MedAgentTrainEnv = train_mod.MedAgentTrainEnv
reward_func      = train_mod.reward_func
build_dataset    = train_mod.build_dataset

DATA_DIR = REPO / "medagentbench_env" / "data"

# Override module paths to point at our cloned data
train_mod._DATA_DIR   = DATA_DIR
train_mod._CACHE_PATH = DATA_DIR / "fhir_cache.json"
train_mod._SYSTEM_PROMPT_PATH = DATA_DIR / "new_system.txt"

# Pre-load shared resources
train_mod._get_mock_fhir()
tasks = train_mod._get_tasks()
print(f"FHIR cache loaded | {len(tasks)} tasks")
print(f"System prompt: {train_mod._get_system_prompt()}...")

FHIR cache loaded | 90 tasks
System prompt: You are an expert medical AI agent.

You will be given a clinical task to perform that involves interacting with a FHIR-compliant EHR system.

Everything you need to complete the task is in the EHR. Do not ask any clarifying questions to the user.

Take your time and think through every step. You MUST plan extensively before each function call, and reflect extensively on the outcomes of the previous function calls.

You have access to the following tools:
- fhir_patient_search: search and filter for patients using FHIR search params
- calculator: evaluate mathematical expressions in python
- fhir_observation_search: search for observations for a patient by code
- fhir_vitals_create: file vital signs for all flowsheets
- fhir_vitals_search: search for vital signs
- fhir_procedure_search: search for procedures
- fhir_condition_search: search for conditions
- fhir_medication_request_create: create a medication request
- fhir_medication_request_s

In [4]:
# Sanity check — run one episode end-to-end
env = MedAgentTrainEnv()
instruction = env.reset()
task = env._task
print(f"Task : {task['id']}  ({task.get('_benchmark_type')})")
print(f"Instr: {instruction[:120]}...")

# Simulate a correct BP observation POST
resp = env.fhir_vitals_create(
    resourceType="Observation",
    category=[{"coding": [{"code": "vital-signs"}]}],
    code={"text": "BP"},
    effectiveDateTime="2023-11-13T10:15:00",
    status="final",
    valueString="118/77",
    subject={"reference": f"Patient/{task['eval_MRN']}"},
)
print(f"POST: {resp[:60]}")

result = env.finish([])
print(f"Finish: {result}")

Task : task3_1  (always-action)
Instr: I just measured the blood pressure for patient with MRN of S2380121, and it is "118/77 mmHg". Help me record it.
Context...
POST: POST request accepted and executed successfully. Please call

────────────────────────────────────────────────────────────
EPISODE TRACE  task=task3_1  steps=2  reward=0.425
────────────────────────────────────────────────────────────
  [1] AGENT: POST http://localhost:8080/fhir/Observation
{"resourceType": "Observation", "status": "final", "category": [{"coding": [{"code": "vital-signs"}]}], "code": {"text": "BP"}, "effectiveDateTime": "2023-11-13T10:15:00", "valueString": "118/77", "subject": {"reference": "Patient/S2380121"}}
  [2] ENV  : POST request accepted and executed successfully. Please call finish if you have got answers for all the questions and finished all the requested tasks
  [3] AGENT: FINISH([])
  [4] ENV  : Task completed.
  ANSWER: []
────────────────────────────────────────────────────────────
Finis

## 4. Build Training Dataset

In [5]:
dataset = build_dataset(DATA_DIR)
print(f"Dataset: {len(dataset)} tasks")
print(f"Roles  : {[m['role'] for m in dataset[0]['prompt']]}")
print(f"User   : {dataset[0]['prompt'][1]['content']}")

Dataset: 90 tasks
Roles  : ['system', 'user']
User   : I just measured the blood pressure for patient with MRN of S2380121, and it is "118/77 mmHg". Help me record it.
Context: It's 2023-11-13T10:15:00+00:00 now. The flowsheet ID for blood pressure is BP.


## 5. Train with GRPOTrainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "./medagent_grpo_output"

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_completion_length=2048,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_steps=10,
    logging_steps=1,
    log_completions=True,
    num_completions_to_print=2,
    fp16=True,   # match Unsloth's fp16 autocaster
    bf16=False,
    save_steps=50,
    save_total_limit=2,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    train_dataset=dataset,
    environment_factory=MedAgentTrainEnv,
    processing_class=tokenizer,
    args=grpo_config,
)

print("Starting GRPO training...")
trainer.train()

## 6. Save Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to {OUTPUT_DIR}")

# Merge LoRA into base weights (optional — needed for full model push)
# merged = model.merge_and_unload()
# merged.save_pretrained(OUTPUT_DIR + "_merged")
# tokenizer.save_pretrained(OUTPUT_DIR + "_merged")

## 7. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import login

HF_TOKEN    = "hf_xxx"          # your HF write token
HF_REPO_ID  = "YOUR_USERNAME/medagent-qwen3-1.7b"

login(token=HF_TOKEN)

# Push LoRA adapter
model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

## 8. Quick Evaluation

Run the trained model on a sample of tasks. The model generates tool calls;
we parse Qwen3's `<tool_call>` format and route to the environment methods.

In [ ]:
import json, re

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (REPO / "medagentbench_env" / "data" / "new_system.txt").read_text().strip()


def parse_tool_call(text: str):
    """Parse Qwen3 <tool_call>{...}</tool_call> format."""
    m = re.search(r'<tool_call>\s*({.*?})\s*</tool_call>', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    # Fallback: bare {"name": ..., "arguments": ...}
    m = re.search(r'\{\s*"name"\s*:\s*"(\w+)"\s*,\s*"arguments"\s*:\s*(\{.*?\})\s*\}',
                  text, re.DOTALL)
    if m:
        try:
            return {"name": m.group(1), "arguments": json.loads(m.group(2))}
        except json.JSONDecodeError:
            pass
    return None


def run_episode(env, max_steps=8):
    """Run one episode with the trained model, return reward and trace."""
    instruction = env.reset()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": instruction},
    ]

    for step in range(max_steps):
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
        reply = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()

        tc = parse_tool_call(reply)
        if tc is None:
            # No parseable tool call — end episode
            env.finish([])
            break

        tool_name = tc.get("name", "")
        tool_args = tc.get("arguments", {})
        method = getattr(env, tool_name, None)

        if method is None or tool_name.startswith("_"):
            env.finish([])
            break

        try:
            tool_result = method(**tool_args)
        except Exception as e:
            tool_result = f"Tool error: {e}"

        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "tool",      "content": tool_result,
                         "tool_call_id": tool_name})

        if env.done:
            break

    return env.reward


# Reset task counter so we evaluate from the start
train_mod._TASK_INDEX = 0

NUM_EVAL = 10
rewards = []

for i in range(NUM_EVAL):
    env = MedAgentTrainEnv()
    r = run_episode(env)
    rewards.append(r)
    print(f"  {env._task['id']}: reward={r:.3f}")

avg = sum(rewards) / len(rewards)
print(f"\nPost-training avg reward ({NUM_EVAL} tasks): {avg:.4f}")
print(f"Baseline (Qwen3-8B OpenRouter):             0.2748")

## 9. Load from HuggingFace (clone)

Re-load the pushed model on any machine — no repo clone needed.

In [ ]:
# from unsloth import FastLanguageModel
# 
# HF_REPO_ID = "YOUR_USERNAME/medagent-qwen3-1.7b"
# 
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=HF_REPO_ID,
#     max_seq_length=4096,
#     load_in_4bit=True,
# )
# FastLanguageModel.for_inference(model)
# print(f"Loaded {HF_REPO_ID} from HuggingFace Hub")